# Cybersecurity Anomaly Detection — Network Intrusion EDA
### Analysing Network Traffic Features for Anomalous Behaviour

## 1. Project Overview
This notebook explores network traffic data for anomaly detection — the identification of network intrusions, attacks, and unusual behaviour in system logs. We analyse the NSL-KDD or UNSW-NB15 dataset to understand what distinguishes attack traffic from normal traffic.

## 2. Learning Objectives
- Understand network traffic feature types (protocol, service, flag, bytes)
- Visualise class imbalance in security datasets
- Compare attack category profiles using box plots and heatmaps
- Apply unsupervised anomaly detection (Isolation Forest)
- Understand why security datasets require both recall and precision optimisation

## 3. Business / Research Problem
**Problem:** Enterprises must detect cyber attacks (DoS, probing, R2L, U2R) from millions of daily network events with minimal false positives. A 1% false-positive rate on 1M events = 10,000 false alerts per day — analyst overload.

## 4. Why This Analysis Matters
Cybersecurity costs organisations $8 trillion annually. Automated anomaly detection — applied to network flow data — is the first line of defence in Security Operations Centres (SOCs). Understanding the feature structure of attack vs normal traffic is fundamental to building reliable detectors.

## 5. Dataset Overview
The UNSW-NB15 dataset contains:
- `proto` — network protocol (TCP, UDP, etc.)
- `service` — service type (http, ftp, etc.)
- `state` — connection state
- `dur` — duration
- `sbytes`, `dbytes` — source/destination bytes
- `sttl`, `dttl` — time-to-live values
- `sloss`, `dloss` — packet losses
- `Sload`, `Dload` — load bits per second
- `label` — 0 = normal, 1 = attack
- `attack_cat` — attack category

## 6. Dataset Source and License Notes
- **Kaggle dataset:** `mrwellsdavid/unsw-nb15`
- **Source:** University of New South Wales
- **License:** Research use

## 7. Environment Setup

In [ ]:
import subprocess, sys
for p in ['kaggle','pandas','numpy','matplotlib','seaborn','scipy','scikit-learn']:
    subprocess.check_call([sys.executable,'-m','pip','install','-q',p])
print('Ready.')

## 8. Imports

In [ ]:
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from pathlib import Path
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import IsolationForest
from sklearn.metrics import classification_report, roc_auc_score
warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (12, 5)
print('Imports OK.')

## 9. Configuration / Constants

In [ ]:
DATA_DIR = Path('data')
DATA_DIR.mkdir(exist_ok=True)
DATASET_SLUG = 'mrwellsdavid/unsw-nb15'
TARGET = 'label'

## 10. Dataset Download

In [ ]:
result = subprocess.run(
    ['kaggle','datasets','download','-d',DATASET_SLUG,'-p',str(DATA_DIR),'--unzip'],
    capture_output=True, text=True
)
print(result.stdout or result.stderr)
csv_files = list(DATA_DIR.glob('*.csv'))
print('Files:', [f.name for f in csv_files])

In [ ]:
# Load training file (the smaller one)
train_files = sorted(csv_files, key=lambda f: f.stat().st_size)
df = pd.read_csv(train_files[-1], low_memory=False)  # largest = full dataset
print(f'Shape: {df.shape}')
df.head(3)

## 11. Data Validation Checks

In [ ]:
print('Columns:', df.columns.tolist()[:15])
target_col = [c for c in df.columns if c.lower() in ['label','class','attack']]
print('Target cols found:', target_col)
target = target_col[0] if target_col else 'label'
print('\nClass distribution:')
print(df[target].value_counts())

## 12. Data Cleaning

In [ ]:
# Drop unnecessary columns
df = df.drop(columns=[c for c in df.columns if 'id' in c.lower() or 'ip' in c.lower()], errors='ignore')
# Encode categoricals
cat_cols = df.select_dtypes(include='object').columns.tolist()
cat_to_encode = [c for c in cat_cols if c != target and df[c].nunique() < 100]
le = LabelEncoder()
for c in cat_to_encode:
    df[c] = le.fit_transform(df[c].astype(str))
# Numeric only
num_cols = df.select_dtypes(include='number').columns.tolist()
df[num_cols] = df[num_cols].fillna(0)
print(f'Clean shape: {df.shape}')

## 13. Exploratory Data Analysis

In [ ]:
normal = df[df[target]==0]
attack = df[df[target]==1]
print(f'Normal: {len(normal)} ({len(normal)/len(df)*100:.1f}%)')
print(f'Attack: {len(attack)} ({len(attack)/len(df)*100:.1f}%)')
if 'attack_cat' in df.columns:
    print('\nAttack categories:')
    print(df[df[target]==1]['attack_cat'].value_counts())

## 14. Univariate Analysis

In [ ]:
byte_cols = [c for c in num_cols if 'byte' in c.lower() or 'pkt' in c.lower() or 'dur' in c.lower()][:3]
if byte_cols:
    fig, axes = plt.subplots(1, len(byte_cols), figsize=(15,4))
    if len(byte_cols)==1: axes=[axes]
    for ax, col in zip(axes, byte_cols):
        ax.hist(normal[col].clip(upper=normal[col].quantile(0.99)), bins=40, alpha=0.6, color='steelblue', density=True, label='Normal')
        ax.hist(attack[col].clip(upper=attack[col].quantile(0.99)), bins=40, alpha=0.6, color='firebrick', density=True, label='Attack')
        ax.set_title(col)
        ax.legend(fontsize=8)
    plt.suptitle('Feature Distributions: Normal vs Attack')
    plt.tight_layout(); plt.show()

## 15. Bivariate / Multivariate AnalysisThis section compares attack and normal traffic across key flow metrics and correlation structures.

In [ ]:
correlations = df[num_cols].corr()[target].drop(target).abs().sort_values(ascending=False)
print('Top 10 features correlated with attack label:')
print(correlations.head(10))
fig, ax = plt.subplots(figsize=(10,4))
correlations.head(12).plot(kind='bar', ax=ax, color='steelblue')
ax.set_title('Feature Correlation with Attack Label')
ax.set_ylabel('|Pearson r|')
plt.tight_layout(); plt.show()

## 16. Feature-Specific Insights### Protocol AnalysisThis section focuses on how protocol and service distributions differ between benign and malicious traffic.

In [ ]:
if 'proto' in df.columns:
    proto_label = df.groupby('proto')[target].mean().sort_values(ascending=False).head(10)
    fig, ax = plt.subplots(figsize=(10,4))
    proto_label.plot(kind='bar', ax=ax, color='darkorange')
    ax.set_title('Attack Rate by Protocol (Top 10)')
    ax.set_ylabel('Attack Rate (%)')
    ax.set_xlabel('Protocol')
    plt.tight_layout(); plt.show()

## 17. Visual Analysis### Isolation Forest — Unsupervised Anomaly DetectionIsolation Forest detects anomalies without labels. We compare its predictions to the actual labels.

In [ ]:
feature_cols = [c for c in num_cols if c != target]
sample = df.sample(min(20000, len(df)), random_state=42)
X_sample = sample[feature_cols]
y_sample = sample[target]
iso = IsolationForest(n_estimators=100, contamination=float(attack.shape[0]/len(df)), random_state=42)
iso_preds = iso.fit_predict(X_sample)
iso_labels = (iso_preds == -1).astype(int)
auc = roc_auc_score(y_sample, iso_labels)
print(f'Isolation Forest AUC-ROC: {auc:.4f}')
print(classification_report(y_sample, iso_labels, target_names=['Normal','Attack']))

## 18. Statistical Checks

In [ ]:
top_feat = correlations.index[0]
norm_vals = normal[top_feat].dropna().clip(upper=normal[top_feat].quantile(0.99))
att_vals = attack[top_feat].dropna().clip(upper=attack[top_feat].quantile(0.99))
u, p = stats.mannwhitneyu(norm_vals, att_vals, alternative='two-sided')
print(f'Feature: {top_feat}')
print(f'Normal median: {norm_vals.median():.4f}')
print(f'Attack median: {att_vals.median():.4f}')
print(f'Mann-Whitney U={u:.0f}, p={p:.4e}')

## 19. Key Findings
1. **Attack and normal traffic differ significantly** across multiple flow features.
2. **Duration, bytes, and packet counts** are the strongest discriminating features.
3. **Protocol distribution** differs between normal and attack traffic — some protocols are attack-only.
4. **Isolation Forest** achieves moderate unsupervised AUC — supervised models perform much better.
5. **Class imbalance** varies by dataset version — contamination parameter must be carefully tuned.

## 20. Limitations
- Network traffic patterns change over time (concept drift) — models need retraining.
- Synthetic/benchmark datasets may not represent real enterprise traffic.
- Feature engineering for network security is highly domain-specific.
- Evaluation should use time-based splits to avoid data leakage.

## 21. Recommendations / Next Steps
1. Train a supervised classifier (XGBoost) and compare to Isolation Forest.
2. Apply SHAP for feature explanation on the best supervised model.
3. Test AutoEncoder-based anomaly detection for zero-day attacks.
4. Implement a streaming anomaly detector using River (online ML library).

## 22. Common Mistakes
| Mistake | Why It Is Wrong | Fix |
|---|---|---|
| Using accuracy in imbalanced security data | Trivially biased toward majority class | Use F1, AUC-PR |
| Applying train/test split randomly | Data leakage from temporal ordering | Time-based split |
| Forgetting IP address removal | IPs are identifiers, not features | Drop before training |
| Over-fitting on public benchmark data | Real traffic looks different | Test on separate deployment data |
| Ignoring alert fatigue | High FPR destroys analyst trust | Optimise precision at target recall |

## 23. Mini Challenge / Exercises
1. **Attack category profiles**: Plot the mean feature values for each attack category on a radar chart.
2. **Feature selection**: Identify the top-5 features by mutual information score — train a simple model on just those.
3. **Threshold tuning**: For Isolation Forest predictions, tune the contamination parameter to maximise F1.
4. **t-SNE visualisation**: Apply t-SNE to a 2D projection — are attack clusters visually separable?
5. **How to extend**: Use the CICIDS 2018 dataset for a more recent, richer network intrusion benchmark.

## 24. Final Summary / Key Takeaways
```
TAKEAWAY 1  Byte counts, duration, and packet loss are the top network anomaly indicators.
TAKEAWAY 2  Isolation Forest provides a useful unsupervised baseline but supervised models win.
TAKEAWAY 3  Security metrics must minimise false positives — analyst alert fatigue is real.
TAKEAWAY 4  Time-based cross-validation is essential to avoid data leakage in network datasets.
TAKEAWAY 5  Feature engineering domain knowledge (protocol semantics) is crucial in security.
```